#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# set device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# choose plug in targets or orthogonal targets
plug_in = False

#### data

In [ ]:
# set dataset parameters
dataset = 'synthetic'
train_size = 1000

In [ ]:
# set confounders
confounders = ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9']
input_dim = len(confounders)

In [ ]:
# read
data_path = ROOT / "data" / "datasets" / f"{dataset}.csv"
df = pd.read_csv(data_path, index_col=0)

#### helpers

In [ ]:
def load_nuisance_models(ns_seed_dir, input_dim, device):
    """Helper to load nuisance models. Set the hidden dim correctly!"""
    paths = {
        "e": ns_seed_dir / "prop_model.pt",
        "m0": ns_seed_dir / "mu0_model.pt",
        "m1": ns_seed_dir / "mu1_model.pt"}

    for name, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing nuisance checkpoint {name}: {path}")

    prop_model = ClassificationHead(input_dim=input_dim, hidden_dim=128).to(device)
    prop_model.load_state_dict(torch.load(paths["e"], map_location=device, weights_only=True))

    m0_model = RegressionHead(input_dim=input_dim, hidden_dim=64).to(device)
    m0_model.load_state_dict(torch.load(paths["m0"], map_location=device, weights_only=True))

    m1_model = RegressionHead(input_dim=input_dim, hidden_dim=64).to(device)
    m1_model.load_state_dict(torch.load(paths["m1"], map_location=device, weights_only=True))

    return prop_model, m0_model, m1_model

#### train and store

In [ ]:
# load configs
config_dir = Path("./configs")
config_name = "plug_in" if plug_in else "orth"
configs = pd.read_csv(config_dir / f"{dataset}_{config_name}.csv", index_col=0)

# set configs
row = configs.iloc[0]
params = dict(
    hidden_dim=int(row["hidden_dim"]),
    learning_rate=float(row["lr"]),
    weight_decay=float(row["weight_decay"]),
    batch_size=int(row["batch_size"]),
    kappa=float(row["kappa"]),
    max_epochs=50,
    patience=5)

In [ ]:
# set directories
out_dir = f'./chkpts/{dataset}/'
os.makedirs(out_dir, exist_ok=True)

ns_dir = ROOT / "experiments" / "nuisances" / "chkpts" / dataset

In [ ]:
# loop over seeds
for seed in range(5):

    # track progress
    print(f" -> Seed {seed}, size {train_size}")
    set_seed(seed)

    # set output dir
    ckpt_dir = Path(out_dir) / f"size_{train_size}" / f"seed_{seed}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # get training data
    _, _, train_df, val_df, _ = make_splits(df=df, train_size=train_size, seed=seed)

    # load nuisance models
    ns_seed_dir = ns_dir / f"size_{train_size}" / f"seed_{seed}"
    prop_model, m0_model, m1_model = load_nuisance_models(ns_seed_dir=ns_seed_dir, input_dim=input_dim, device=device)

    # add pseudo outcomes to dataframes
    train_df = compute_dr_scores(train_df, confounders, prop_model, m0_model, m1_model, device)
    val_df = compute_dr_scores(val_df, confounders, prop_model, m0_model, m1_model, device)

    # make data loaders
    train_loader, _ = make_ranker_loaders(train_df, val_df, confounders, params['kappa'], params['batch_size'])
    _, val_loader = make_cate_loaders(train_df, val_df, confounders, params['batch_size'])

    # init model
    ranker = ClassificationHead(input_dim, hidden_dim=params['hidden_dim']).to(device)
    
    # train model
    ranker, info = train_ranker(ranker, train_loader, val_loader, device, lr=params['learning_rate'], weight_decay=params['weight_decay'],
                                      max_epochs=50, patience=5, seed=seed, fraction_of_pairs=0.1, plug_in=plug_in)

    # checkpoint
    method_name = "plug_in" if plug_in else "orthogonal"
    torch.save(ranker.state_dict(), ckpt_dir / f"{method_name}.pt")